# Notebook 2 of 5: Baselines
### Swahili Tweet Sentiment Classification (AfriSenti)

This notebook:
1. Sets up the project folder (the fixed split from notebook 1 is already in it)
2. Sets two reference points that need no learning: always predict the most common class, and a random guess in proportion to the class frequencies
3. Trains **TF-IDF + Logistic Regression** with three kinds of features (words, characters, both), tuned on the **validation** set
4. Evaluates the chosen model once on the **test** set
5. Saves metrics, a confusion matrix, errors and the top words per class

**Why these reference points:** in a three-class problem where 59% of tweets are neutral, a model that always says "neutral" already gets 59% accuracy. A model is only useful if it clearly beats that, and macro-F1 shows it honestly.

## Step 0 - Set up the project folder

Pick the `.venv` kernel (top right of the notebook). The libraries come from `requirements.txt` (`pip install -r requirements.txt` in the `.venv`).

The next cell moves to the project root. Everything this notebook writes goes inside the project folder, so there is nothing to clone or upload.

In [ ]:
from getpass import getpass
from pathlib import Path
import os

# VS Code starts a notebook inside its own folder (notebooks/). Move to the project root so the
# relative paths used below (data/, figures/, results/ ...) resolve from there.
ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir(), "Open the project folder in VS Code and run this notebook from inside it."
os.chdir(ROOT)
print("Project folder:", ROOT)

for folder in ["results/confusion_matrices", "results/metrics", "results/errors", "results/predictions", "figures/baseline"]:
    os.makedirs(folder, exist_ok=True)

## Step 1 - Load the fixed split

In [ ]:
import pandas as pd, numpy as np

LABELS = ["negative", "neutral", "positive"]
train = pd.read_csv("data/processed/train.csv")
val = pd.read_csv("data/processed/val.csv")
test = pd.read_csv("data/processed/test.csv")
for df in (train, val, test):
    df["text_clean"] = df["text_clean"].fillna("").astype(str)
print("train:", len(train), " val:", len(val), " test:", len(test))
print(train["label"].value_counts().to_dict())

## Step 2 - Metrics helper

Primary metric: **macro-F1** (all three classes count equally). Also reported: accuracy, macro-precision, macro-recall, and **weighted F1**, which is the metric the SemEval shared task used.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def scores(y_true, pred):
    return {
        "accuracy": accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro"),
        "weighted_f1": f1_score(y_true, pred, average="weighted"),
        "precision_macro": precision_score(y_true, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, pred, average="macro", zero_division=0),
    }

def show(name, s):
    print(f"{name:34s} acc {s['accuracy']:.4f}  macro-F1 {s['macro_f1']:.4f}  weighted-F1 {s['weighted_f1']:.4f}")

y_tr, y_va, y_te = train["label_id"].values, val["label_id"].values, test["label_id"].values

## Step 3 - Two reference points

Both ignore the text completely.

In [ ]:
from sklearn.dummy import DummyClassifier

ref_rows = []
for name, strategy in [("Majority class (always neutral)", "most_frequent"), ("Random by class frequency", "stratified")]:
    d = DummyClassifier(strategy=strategy, random_state=42).fit(train["text_clean"], y_tr)
    s_val = scores(y_va, d.predict(val["text_clean"]))
    s_te = scores(y_te, d.predict(test["text_clean"]))
    show(name + " [val]", s_val); show(name + " [test]", s_te)
    ref_rows.append((name, s_te))

## Step 4 - TF-IDF + Logistic Regression, tuned on validation

TF-IDF turns each tweet into a vector of weighted word counts. Logistic Regression learns which features push a tweet towards each class. `class_weight="balanced"` makes the small negative class count more, because the classes are imbalanced. `C` controls regularisation.

Three feature sets are compared, because Swahili adds prefixes and suffixes to words and tweets contain typos: **word** 1 to 2-grams, **character** 2 to 5-grams (inside word boundaries), and **both together**. Every combination is scored on the validation set only. The test set is not touched here.

In [ ]:
import time
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def make_vecs(kind):
    vecs = []
    if kind in ("word", "word+char"):
        vecs.append(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1, max_features=100000))
    if kind in ("char", "word+char"):
        vecs.append(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2, max_features=200000))
    return vecs

def featurize(kind):
    vecs = make_vecs(kind)
    Xtr = hstack([v.fit_transform(train["text_clean"]) for v in vecs]).tocsr()    # fit on train only
    Xva = hstack([v.transform(val["text_clean"]) for v in vecs]).tocsr()
    Xte = hstack([v.transform(test["text_clean"]) for v in vecs]).tocsr()
    return vecs, Xtr, Xva, Xte

grid, feats = [], {}
for kind in ["word", "char", "word+char"]:
    feats[kind] = featurize(kind)
    _, Xtr, Xva, _ = feats[kind]
    for C in [0.1, 0.5, 1, 5, 10, 50]:
        clf = LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        clf.fit(Xtr, y_tr)
        f1 = f1_score(y_va, clf.predict(Xva), average="macro")
        grid.append({"features": kind, "C": C, "val_macro_f1": round(f1, 4)})
        print(f"{kind:10s} C={C:<5} val macro-F1 = {f1:.4f}")

grid_df = pd.DataFrame(grid)
grid_df.to_csv("results/metrics/baseline_val_grid.csv", index=False)
best = grid_df.sort_values("val_macro_f1", ascending=False).iloc[0]
print()
print("Best on validation:", best.to_dict())
print()
print(grid_df.pivot(index="C", columns="features", values="val_macro_f1"))

## Step 5 - Train the final model and evaluate on the test set (once)

In [ ]:
best_kind, best_C = best["features"], float(best["C"])
vecs, Xtr, Xva, Xte = feats[best_kind]

t0 = time.time()
final = LogisticRegression(C=best_C, max_iter=3000, class_weight="balanced").fit(Xtr, y_tr)
train_time = time.time() - t0
pred = final.predict(Xte)

s_te = scores(y_te, pred)
for k, v in s_te.items():
    print(f"{k:16s} {v:.4f}")
print(f"train time (final fit only): {train_time:.1f}s\n")
print(classification_report(y_te, pred, target_names=LABELS, zero_division=0))

## Step 6 - Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_te, pred, normalize="true")
fig, ax = plt.subplots(figsize=(5.5, 4.8))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_title("TF-IDF + Logistic Regression: normalised confusion matrix (test)")
plt.tight_layout()
plt.savefig("results/confusion_matrices/cm_baseline_tfidf_logreg.png", dpi=150, bbox_inches="tight")
plt.show()
print(pd.DataFrame(confusion_matrix(y_te, pred), index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS]))

## Step 7 - Save metrics, predictions, errors and top features

The predictions file lets later notebooks compare models tweet by tweet (and test whether differences are bigger than chance). The top features show *what the model learned*. You read Swahili, so check whether they make sense.

In [ ]:
import os

rows = []
for name, s in ref_rows:
    rows.append({"model_name": name, **{k: round(v, 4) for k, v in s.items()}, "macro_f1_std": "", "train_time_s": 0, "notes": "no learning"})
rows.append({"model_name": "TF-IDF + LogReg", **{k: round(v, 4) for k, v in s_te.items()}, "macro_f1_std": "",
             "train_time_s": round(train_time, 1), "notes": f"features={best_kind}, C={best_C}, class_weight=balanced"})
summary = pd.DataFrame(rows)
summary.to_csv("results/metrics_summary.csv", index=False)
print(summary[["model_name", "accuracy", "macro_f1", "weighted_f1"]])

pd.DataFrame({"id": test["id"], "true": y_te, "pred": pred}).to_csv("results/predictions/tfidf_test_preds.csv", index=False)

errs = test.assign(pred=pred)
errs = errs[errs["label_id"] != errs["pred"]].copy()
errs["pred_name"] = errs["pred"].map(lambda i: LABELS[i])
errs[["id", "text", "label", "pred_name"]].to_csv("results/errors/baseline_errors.csv", index=False)
print("\nMisclassified:", len(errs), "of", len(test))
print(errs.groupby(["label", "pred_name"]).size().sort_values(ascending=False))

# top features per class (only the word features are readable)
if best_kind in ("word", "word+char"):
    names = np.array(vecs[0].get_feature_names_out())
    n_word = len(names)
    coefs = final.coef_[:, :n_word]
    top = {LABELS[i]: ", ".join(names[np.argsort(coefs[i])[-12:][::-1]]) for i in range(3)}
    pd.DataFrame({"class": list(top), "top_words": list(top.values())}).to_csv("results/metrics/baseline_top_words.csv", index=False)
    print()
    for k, v in top.items():
        print(f"{k:9s} {v}")

## Last step - Save your work

Everything this notebook wrote is already in the project folder (`results/` and `figures/`). Save the notebook itself (Cmd+S).

## Done - notebook 2

Paste these back to me:
- Step 3: the two reference scores
- Step 4: the full grid and the best setting
- Step 5: the test metrics and the per-class report
- Step 6: the confusion counts
- Step 7: the misclassification counts and the top words